In [ ]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path

ROOT = Path.cwd()
RAW_OFFERS = ROOT / "data" / "raw" / "offers"

# change RAW_OFFERS per absolute path if the file data is not in the working dir
# ex :RAW_OFFERS = Path("/Users/../.../data/raw/offers")


if not RAW_OFFERS.exists():
    raise FileNotFoundError("offers file not found")

In [ ]:
def build_auctions(offers_path: str | Path = RAW_OFFERS, product: str = "mFRR", delay_hours: float = 2) -> pd.DataFrame:
    table = pq.read_table(offers_path, filters=[
                          ("auction_type", "==", product)])
    offers = table.to_pandas()
    offers = offers[~offers["is_anticipated"]].drop(columns=["is_anticipated"])
    accepted_offers = offers[offers["accepted"]]
    auctions = offers.groupby('auction_id').agg(
        date=('date', 'first'), direction=('direction', 'first'),
        freq=('auction_frequency', 'first'), block=('time_block', 'first'),
        gc=('gate_closure_time_utc', 'first'), n_bids=('power_price', 'size'),
        offered=('offered_volume', 'sum'), awarded=('awarded_volume', 'sum'),
        holiday_share=('holiday_share', 'first'), day_of_week=('day_of_week', 'first'),
        day_type=('day_type', 'first'))

    # nb of accepted bids=> 0 for no-buy auctions
    auctions['n_accepted'] = accepted_offers.groupby('auction_id').size().reindex(
        auctions.index, fill_value=0)
    # offered / awarded volume, Nan when nothing was bought
    auctions['supply_ratio'] = auctions.offered / \
        auctions.awarded.where(auctions.awarded > 0)
    auctions['marginal'] = accepted_offers.groupby(
        # regroups bids accepted per auction
        "auction_id")["power_price"].max()
    auctions['y'] = np.log10(auctions.marginal)
    auctions['avail'] = auctions.gc + pd.Timedelta(hours=delay_hours)
    auctions['series'] = auctions.freq + '|' + \
        auctions.direction + '|' + \
        auctions.block.where(auctions.freq != 'weekly',
                             'week')  # not supposed to have nans but never know

    return auctions.reset_index().sort_values('gc').reset_index(drop=True)  # sort on gc

In [ ]:
a = build_auctions()

In [ ]:
a.head()

,auction_id,date,direction,freq,block,gc,n_bids,offered,awarded,holiday_share,day_of_week,day_type,n_accepted,supply_ratio,marginal,y,avail,series
0,TRL+_19_01_01_00_04,2019-01-01,upwards,daily,00_04,2018-12-28 14:30:00+00:00,22,752,183,1.0,1,sunday_holiday,5,4.109290,0.58,-0.236572,2018-12-28 16:30:00+00:00,daily|upwards|00_04
1,TRL-_19_01_01_08_12,2019-01-01,downwards,daily,08_12,2018-12-28 14:30:00+00:00,15,467,20,1.0,1,sunday_holiday,2,23.350000,2.85,0.454845,2018-12-28 16:30:00+00:00,daily|downwards|08_12
2,TRL-_19_01_01_12_16,2019-01-01,downwards,daily,12_16,2018-12-28 14:30:00+00:00,15,462,15,1.0,1,sunday_holiday,2,30.800000,1.25,0.096910,2018-12-28 16:30:00+00:00,daily|downwards|12_16
3,TRL-_19_01_01_16_20,2019-01-01,downwards,daily,16_20,2018-12-28 14:30:00+00:00,16,467,15,1.0,1,sunday_holiday,2,31.133333,4.00,0.602060,2018-12-28 16:30:00+00:00,daily|downwards|16_20
4,TRL-_19_01_01_20_24,2019-01-01,downwards,daily,20_24,2018-12-28 14:30:00+00:00,15,462,15,1.0,1,sunday_holiday,2,30.800000,1.40,0.146128,2018-12-28 16:30:00+00:00,daily|downwards|20_24


In [ ]:
# a[a["auction_id"] == "TRL+_24_07_15_12_16", "awarded"].iloc[0]

a.loc[a["auction_id"] == "TRL+_24_07_15_12_16", "awarded"].iloc[0]

np.int64(0)

In [ ]:


# sort by series then auction_id (stack consecutive same type series)
a_sorted = a.sort_values(["series", "auction_id"]).reset_index(drop=True)

iszero = (a_sorted["awarded"] == 0)


change = iszero.ne(iszero.shift()) | a_sorted["series"].ne(
    # increasing for change of state or change of series
    a_sorted["series"].shift())
grp = change.cumsum()


lignes = []
for _, g in a_sorted[iszero].groupby(grp[iszero]):  # only keep 0 and regroup
    longueur = len(g)
    if longueur > 10:   # sequences w/ len>10
        lignes.append({
            "series":   g["series"].iloc[0],
            "beg":    g["auction_id"].iloc[0],
            "end":      g["auction_id"].iloc[-1],
            "length": longueur,
        })

resultat = pd.DataFrame(lignes)
resultat

,series,beg,end,length
0,daily|downwards|00_04,TRL-_19_08_28_00_04,TRL-_19_09_08_00_04,12
1,daily|downwards|00_04,TRL-_21_08_09_00_04,TRL-_21_08_29_00_04,21
2,daily|downwards|00_04,TRL-_22_02_07_00_04,TRL-_22_02_20_00_04,14
3,daily|downwards|00_04,TRL-_22_07_18_00_04,TRL-_22_07_31_00_04,14
4,daily|downwards|00_04,TRL-_23_03_20_00_04,TRL-_23_04_16_00_04,28
5,daily|downwards|00_04,TRL-_24_01_01_00_04,TRL-_24_12_01_00_04,305
6,daily|downwards|04_08,TRL-_21_08_09_04_08,TRL-_21_08_29_04_08,21
7,daily|downwards|04_08,TRL-_22_02_07_04_08,TRL-_22_02_20_04_08,14
8,daily|downwards|04_08,TRL-_22_07_18_04_08,TRL-_22_07_31_04_08,14
9,daily|downwards|04_08,TRL-_23_03_20_04_08,TRL-_23_04_16_04_08,28
